
# Float64 学习型迭代优化器：SGD 与 Adam 两个候选方案

本 Notebook 将一个简单的隐式欧拉时间步求解问题整理为可复现、可阅读、可修改的教学案例。

我们只保留前序消融实验中表现最好的两个候选方案：

| 方案名称 | PyTorch 优化器 | 学习率 |
|---|---:|---:|
| `sgd_lr_1e-2` | SGD，无 momentum | $10^{-2}$ |
| `adam_lr_1e-4` | Adam，默认 `betas=(0.9, 0.999)` | $10^{-4}$ |

Notebook **不会自动遍历两个方案**。在最后一个单元格中修改 `SELECTED_EXPERIMENT`，每次只运行一个实验。这样更适合逐段阅读代码、观察输出和调试。

固定条件：

- 数值精度：`torch.float64`
- 训练数据：精确初值 $y_0$ 加上初值附近 10 个随机扰动点
- 扰动标准差：$\sigma=10^{-2}$
- 输入归一化：启用
- 输出缩放：网络输出乘以 $\Delta t$
- MLP：两层隐藏层，每层 32 个神经元，激活函数为 ReLU
- 最后一层：零初始化

> 公式渲染说明：块级公式统一使用 `$$ ... $$`，以兼容 JupyterLab、经典 Notebook 和常见 Markdown 渲染器。



## 1. 问题定义：自由落体的一步隐式欧拉更新

考虑一个质量为 $m$ 的质点。在时刻 $t_n$，已知位置和速度：

$$
p_n\in\mathbb{R}^3,
\qquad
v_n\in\mathbb{R}^3.
$$

重力方向沿 $z$ 轴负方向。我们希望求解下一时刻的位置：

$$
y = p_{n+1}.
$$

对自由落体使用隐式欧拉离散，可以把下一时刻位置写成一个变分能量最小化问题：

$$
y^\ast
=
\arg\min_y E(y),
$$

其中：

$$
E(y)
=
\frac{m}{2\Delta t^2}
\left\|
y-p_n-\Delta t\,v_n
\right\|_2^2
+
mg\,y_z.
$$

这个问题是严格凸二次优化问题，因此存在唯一最优解：

$$
y^\ast
=
p_n
+
\Delta t\,v_n
-
\Delta t^2
\begin{bmatrix}
0\\
0\\
g
\end{bmatrix}.
$$

由于 Hessian 为常数矩阵：

$$
\nabla^2 E(y)
=
\frac{m}{\Delta t^2}I,
$$

Newton 法从任意初值出发都可以一步到达理论最优解。它是本实验的参考基准。


In [ ]:

from __future__ import annotations

import json
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

TORCH_DTYPE = torch.float64
torch.set_default_dtype(TORCH_DTYPE)

# float64 下允许观察更小的 gap 和 residual。
PLOT_FLOOR = 1e-30

print("PyTorch version:", torch.__version__)
print("Default dtype:", torch.get_default_dtype())



## 2. 实验配置

为了避免把“代码阅读”和“批量实验管理”混在一起，这里只定义两个候选方案。运行时由 `main()` 根据字符串名称选取一个方案。

训练集固定为：

$$
\mathcal{D}
=
\left\{
y_0,\;
y_0+\sigma\epsilon_1,\;
\dots,\;
y_0+\sigma\epsilon_{10}
\right\},
\qquad
\epsilon_i\sim\mathcal{N}(0,I),
\qquad
\sigma=10^{-2}.
$$

两个方案复用完全相同的训练点、模型初始化种子和物理参数。唯一变化是：用于训练 MLP 参数的 PyTorch 优化器不同。


In [ ]:

@dataclass(frozen=True)
class PhysicsConfig:
    m: float = 1.0
    g: float = 9.8
    dt: float = 0.01
    p_n: Tuple[float, float, float] = (3.0, 4.0, 5.0)
    v_n: Tuple[float, float, float] = (0.5, -0.5, 0.0)


@dataclass(frozen=True)
class TrainingConfig:
    epochs: int = 1000
    initial_k: int = 1
    k_increase_interval: int = 200
    max_k: int = 10
    eval_interval: int = 100
    eval_steps: int = 10
    final_test_steps: int = 15

    num_perturbation_points: int = 10
    perturbation_std: float = 1e-2
    perturbation_seed: int = 123
    model_seed: int = 42

    use_input_normalization: bool = True
    use_dt_scaling: bool = True


@dataclass(frozen=True)
class ExperimentConfig:
    name: str
    optimizer_name: str
    learning_rate: float
    description: str


PHYSICS_CONFIG = PhysicsConfig()
TRAINING_CONFIG = TrainingConfig()

EXPERIMENTS: Dict[str, ExperimentConfig] = {
    "sgd_lr_1e-2": ExperimentConfig(
        name="sgd_lr_1e-2",
        optimizer_name="sgd",
        learning_rate=1e-2,
        description="Float64 + SGD without momentum + learning rate 1e-2",
    ),
    "adam_lr_1e-4": ExperimentConfig(
        name="adam_lr_1e-4",
        optimizer_name="adam",
        learning_rate=1e-4,
        description="Float64 + Adam with default betas + learning rate 1e-4",
    ),
}

OUTPUT_ROOT = Path.cwd() / "learned_optimizer_float64_teaching_outputs"

print("Available experiments:")
for key, config in EXPERIMENTS.items():
    print(f"  {key}: {config.description}")



## 3. 学习型迭代器

传统 Newton 法根据解析梯度和 Hessian 构造更新方向。这里希望训练一个 MLP，让它直接预测迭代增量：

$$
y^{(k+1)}
=
y^{(k)}
+
\Delta y_\theta^{(k)}.
$$

网络输入由三部分拼接而成：

$$
\text{input}
=
\left[
y^{(k)},\;
p_n,\;
v_n,\;
m,\;
g,\;
\Delta t
\right]
\in\mathbb{R}^{12}.
$$

模型实际使用归一化输入，并将网络原始输出乘以 $\Delta t$：

$$
\Delta y_\theta
=
\Delta t\cdot
\operatorname{MLP}_\theta
\left(
\frac{\text{input}-\mu}{s}
\right).
$$

输出乘以 $\Delta t$ 的作用是给更新量提供一个与时间步长一致的自然尺度。

最后一层采用零初始化。因此，在训练开始前：

$$
\Delta y_\theta = 0.
$$

这可以避免随机初始化直接产生过大的位移更新。


In [ ]:

class MLPOptimizer(nn.Module):
    """预测单步位置增量的学习型迭代器。"""

    def __init__(
        self,
        input_mean: torch.Tensor,
        input_std: torch.Tensor,
        use_input_normalization: bool = True,
        use_dt_scaling: bool = True,
    ) -> None:
        super().__init__()

        self.use_input_normalization = use_input_normalization
        self.use_dt_scaling = use_dt_scaling

        self.net = nn.Sequential(
            nn.Linear(12, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 3),
        )

        # 初始状态下网络输出恒为零。
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

        self.register_buffer("input_mean", input_mean.clone().detach())
        self.register_buffer("input_std", input_std.clone().detach())

    def forward(
        self,
        y: torch.Tensor,
        history: torch.Tensor,
        params: torch.Tensor,
    ) -> torch.Tensor:
        network_input = torch.cat([y, history, params], dim=-1)

        if self.use_input_normalization:
            network_input = (
                network_input - self.input_mean
            ) / self.input_std

        delta = self.net(network_input)

        if self.use_dt_scaling:
            dt = params[2]
            delta = dt * delta

        return delta



## 4. 理论解、Newton 法和驻点残差

能量函数的一阶导数为：

$$
\nabla E(y)
=
\frac{m}{\Delta t^2}
\left(
y-p_n-\Delta t\,v_n
\right)
+
\begin{bmatrix}
0\\
0\\
mg
\end{bmatrix}.
$$

定义驻点残差：

$$
r(y)=\nabla E(y).
$$

理论最优解满足：

$$
r(y^\ast)=0.
$$

Newton 方向为：

$$
\Delta y_{\text{Newton}}
=
-
\left(
\nabla^2 E(y)
\right)^{-1}
\nabla E(y).
$$

对于当前严格凸二次问题：

$$
\nabla^2 E(y)
=
\frac{m}{\Delta t^2}I.
$$

因此 Newton 法一步即可到达 $y^\ast$。


In [ ]:

def variational_energy(
    y: torch.Tensor,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    m: float,
    g: float,
    dt: float,
) -> torch.Tensor:
    """隐式欧拉变分能量 E(y)。"""
    inertial_residual = y - p_n - dt * v_n
    kinetic_term = (m / (2.0 * dt**2)) * torch.sum(inertial_residual**2)
    potential_term = m * g * y[2]
    return kinetic_term + potential_term


def stationarity_residual(
    y: torch.Tensor,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    m: float,
    g: float,
    dt: float,
) -> torch.Tensor:
    """一阶驻点方程残差 r(y) = grad E(y)。"""
    residual = (m / dt**2) * (y - p_n - dt * v_n)
    residual = residual.clone()
    residual[2] += m * g
    return residual


def stationarity_residual_norm(
    y: torch.Tensor,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    m: float,
    g: float,
    dt: float,
) -> torch.Tensor:
    return torch.norm(stationarity_residual(y, p_n, v_n, m, g, dt))


def exact_solution(
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    g: float,
    dt: float,
) -> torch.Tensor:
    gravity = torch.tensor([0.0, 0.0, g], dtype=p_n.dtype, device=p_n.device)
    return p_n + dt * v_n - dt**2 * gravity


def newton_direction(
    y: torch.Tensor,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    m: float,
    g: float,
    dt: float,
) -> torch.Tensor:
    gradient = stationarity_residual(y, p_n, v_n, m, g, dt)
    inverse_hessian_scale = dt**2 / m
    return -inverse_hessian_scale * gradient



## 5. 训练数据：在原始初值附近扰动

原始优化变量初值取为：

$$
y_0=p_n.
$$

训练集除了保留精确初值 $y_0$，还会加入 10 个局部扰动点：

$$
y_i
=
y_0+\sigma\epsilon_i,
\qquad
\epsilon_i\sim \mathcal{N}(0,I),
\qquad
\sigma=10^{-2}.
$$

保留精确的 $y_0$ 很重要，因为最终测试也统一从 $y_0$ 出发。

对网络输入进行归一化时，只使用训练集统计量。由于 `history=[p_n,v_n]` 和 `params=[m,g,dt]` 在当前单一物理问题中保持不变，它们的标准差会接近零。代码会把过小标准差替换为 1，避免除零。


In [ ]:

def make_training_states_near_initial(
    y0: torch.Tensor,
    perturbation_std: float,
    num_perturbation_points: int,
    seed: int,
) -> List[torch.Tensor]:
    """构造 y0 与 y0 附近的随机扰动点。"""
    states = [y0.clone()]

    generator = torch.Generator(device=y0.device)
    generator.manual_seed(seed)

    for _ in range(num_perturbation_points):
        noise = torch.randn(
            3,
            generator=generator,
            dtype=y0.dtype,
            device=y0.device,
        )
        states.append(y0 + perturbation_std * noise)

    return states


def compute_input_normalizer(
    training_states: List[torch.Tensor],
    history: torch.Tensor,
    params: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    inputs = torch.stack(
        [torch.cat([y, history, params], dim=-1) for y in training_states],
        dim=0,
    )

    mean = inputs.mean(dim=0)
    std = inputs.std(dim=0, unbiased=False)

    # 当前任务中，history 与 params 不随数据点变化。
    # 这些分量的 std 会为零，因此替换为 1。
    std = torch.where(std < 1e-12, torch.ones_like(std), std)
    return mean, std


def plot_training_states(
    training_states: List[torch.Tensor],
    y0: torch.Tensor,
    y_star: torch.Tensor,
    save_path: Path,
) -> None:
    points = torch.stack(training_states).detach().cpu().numpy()
    y0_np = y0.detach().cpu().numpy()
    y_star_np = y_star.detach().cpu().numpy()

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(points[:, 0], points[:, 1], points[:, 2], label="Training states")
    ax.scatter(*y0_np, marker="x", s=100, label="Exact initial state y0")
    ax.scatter(*y_star_np, marker="*", s=180, label="Exact solution y*")

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title("Training states near the original initial point")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)



## 6. 评价指标

仅观察能量值还不够。当前问题中，Newton 法一步即可到达精确解，所以我们希望同时检查以下指标。

### 6.1 能量 gap

$$
\operatorname{gap}(y)
=
E(y)-E(y^\ast).
$$

对于当前二次问题：

$$
E(y)-E(y^\ast)
=
\frac{m}{2\Delta t^2}
\left\|
y-y^\ast
\right\|_2^2.
$$

它衡量当前位置与最优点之间的能量差。

### 6.2 驻点 residual

$$
\operatorname{residual}(y)
=
\left\|
\nabla E(y)
\right\|_2.
$$

它直接衡量一阶最优性条件是否满足。由于 residual 对微小位置误差更加敏感，所以它通常比 loss gap 更适合区分两个已经非常接近最优解的方案。

### 6.3 位置误差

$$
\operatorname{position\ error}(y)
=
\left\|
y-y^\ast
\right\|_2.
$$

这个指标最直观：它直接表示迭代点与理论解之间的欧氏距离。

### 6.4 更新步长

$$
\left\|
\Delta y^{(k)}
\right\|_2.
$$

如果迭代器已经稳定收敛，更新步长应逐渐减小。

### 6.5 周期性冻结评估

训练期间，网络参数会持续变化。为了避免把一个 epoch 内不同网络状态下的 loss 混在一起，我们每隔若干 epoch 冻结一次当前模型，并从全部训练初值分别运行固定步数，记录其中最差的最终 gap 和 residual。


In [ ]:

def compute_metrics(
    y: torch.Tensor,
    y_star: torch.Tensor,
    E_star: float,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    m: float,
    g: float,
    dt: float,
) -> Dict[str, float]:
    energy = variational_energy(y, p_n, v_n, m, g, dt).item()
    return {
        "loss": float(energy),
        "gap": max(float(energy - E_star), 0.0),
        "residual_norm": float(
            stationarity_residual_norm(y, p_n, v_n, m, g, dt).item()
        ),
        "position_error": float(torch.norm(y - y_star).item()),
    }


def evaluate_rollout(
    model: MLPOptimizer,
    initial_y: torch.Tensor,
    history: torch.Tensor,
    params: torch.Tensor,
    y_star: torch.Tensor,
    E_star: float,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    m: float,
    g: float,
    dt: float,
    num_steps: int,
) -> List[Dict[str, float]]:
    """冻结网络参数，从指定初值出发运行固定步数。"""
    y = initial_y.clone()
    records: List[Dict[str, float]] = []

    initial_metrics = compute_metrics(y, y_star, E_star, p_n, v_n, m, g, dt)
    records.append({"step": 0, "update_norm": 0.0, **initial_metrics})

    for step in range(1, num_steps + 1):
        with torch.no_grad():
            delta = model(y, history, params)
            y = y + delta

        metrics = compute_metrics(y, y_star, E_star, p_n, v_n, m, g, dt)
        records.append(
            {
                "step": step,
                "update_norm": float(torch.norm(delta).item()),
                **metrics,
            }
        )

    return records


def evaluate_newton_rollout(
    initial_y: torch.Tensor,
    y_star: torch.Tensor,
    E_star: float,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    m: float,
    g: float,
    dt: float,
    num_steps: int,
) -> List[Dict[str, float]]:
    y = initial_y.clone()
    records: List[Dict[str, float]] = []

    initial_metrics = compute_metrics(y, y_star, E_star, p_n, v_n, m, g, dt)
    records.append({"step": 0, "update_norm": 0.0, **initial_metrics})

    for step in range(1, num_steps + 1):
        delta = newton_direction(y, p_n, v_n, m, g, dt)
        y = y + delta

        metrics = compute_metrics(y, y_star, E_star, p_n, v_n, m, g, dt)
        records.append(
            {
                "step": step,
                "update_norm": float(torch.norm(delta).item()),
                **metrics,
            }
        )

    return records



## 7. 训练策略

每个训练 epoch 中，会从训练集中的每个初值分别出发，执行 $K$ 步迭代。

单个微步的过程为：

1. 当前迭代点为 $y^{(k)}$；
2. MLP 预测 $\Delta y_\theta^{(k)}$；
3. 更新：$y^{(k+1)} = y^{(k)} + \Delta y_\theta^{(k)}$；
4. 计算：$E\left(y^{(k+1)}\right)$；
5. 对网络参数执行一次 `backward()` 和 `optimizer.step()`；
6. 对 $y^{(k+1)}$ 执行 `detach()`，再进入下一步。

这里采用的是**逐微步在线更新**，而不是把全部训练点的 loss 累加后统一反向传播。

为了逐渐训练多步迭代能力，rollout 深度 $K$ 会随 epoch 增长：

$$
K =
1,2,3,\dots
$$

默认每 200 个 epoch 增加 1，最大值为 10。


In [ ]:

def create_optimizer(
    model: nn.Module,
    experiment: ExperimentConfig,
) -> torch.optim.Optimizer:
    name = experiment.optimizer_name.lower()

    if name == "sgd":
        # 不使用 momentum，减少额外变量。
        return torch.optim.SGD(
            model.parameters(),
            lr=experiment.learning_rate,
        )

    if name == "adam":
        return torch.optim.Adam(
            model.parameters(),
            lr=experiment.learning_rate,
        )

    raise ValueError(f"Unsupported optimizer: {experiment.optimizer_name!r}")


def frozen_evaluate_training_set(
    model: MLPOptimizer,
    training_states: List[torch.Tensor],
    history: torch.Tensor,
    params: torch.Tensor,
    y_star: torch.Tensor,
    E_star: float,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    m: float,
    g: float,
    dt: float,
    num_steps: int,
) -> Dict[str, float]:
    """冻结当前模型，对全部训练初值执行统一评估。"""
    final_records = []

    for initial_y in training_states:
        rollout = evaluate_rollout(
            model=model,
            initial_y=initial_y,
            history=history,
            params=params,
            y_star=y_star,
            E_star=E_star,
            p_n=p_n,
            v_n=v_n,
            m=m,
            g=g,
            dt=dt,
            num_steps=num_steps,
        )
        final_records.append(rollout[-1])

    return {
        "worst_gap": max(record["gap"] for record in final_records),
        "mean_gap": float(np.mean([record["gap"] for record in final_records])),
        "worst_residual_norm": max(
            record["residual_norm"] for record in final_records
        ),
        "mean_residual_norm": float(
            np.mean([record["residual_norm"] for record in final_records])
        ),
    }


def train_selected_model(
    model: MLPOptimizer,
    optimizer: torch.optim.Optimizer,
    training_states: List[torch.Tensor],
    history: torch.Tensor,
    params: torch.Tensor,
    y_star: torch.Tensor,
    E_star: float,
    p_n: torch.Tensor,
    v_n: torch.Tensor,
    physics: PhysicsConfig,
    training: TrainingConfig,
) -> Dict[str, List[Dict[str, float]]]:
    """训练一个被选中的候选方案。"""
    online_log: List[Dict[str, float]] = []
    frozen_log: List[Dict[str, float]] = []

    k = training.initial_k

    for epoch in range(training.epochs):
        if (
            epoch > 0
            and epoch % training.k_increase_interval == 0
            and k < training.max_k
        ):
            k += 1

        trajectory_final_gaps: List[float] = []
        micro_step_gaps: List[float] = []

        for initial_y in training_states:
            y = initial_y.clone()

            for _ in range(k):
                delta = model(y, history, params)
                y = y + delta

                loss = variational_energy(
                    y,
                    p_n,
                    v_n,
                    physics.m,
                    physics.g,
                    physics.dt,
                )

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

                gap = max(float(loss.item() - E_star), 0.0)
                micro_step_gaps.append(gap)

                # 不跨 rollout 步回传梯度。
                y = y.detach()

            trajectory_final_gaps.append(gap)

        online_log.append(
            {
                "epoch": epoch,
                "k": k,
                "worst_trajectory_final_gap": max(trajectory_final_gaps),
                "mean_trajectory_final_gap": float(
                    np.mean(trajectory_final_gaps)
                ),
                "worst_micro_step_gap": max(micro_step_gaps),
            }
        )

        should_evaluate = (
            epoch % training.eval_interval == 0
            or epoch == training.epochs - 1
        )

        if should_evaluate:
            frozen_metrics = frozen_evaluate_training_set(
                model=model,
                training_states=training_states,
                history=history,
                params=params,
                y_star=y_star,
                E_star=E_star,
                p_n=p_n,
                v_n=v_n,
                m=physics.m,
                g=physics.g,
                dt=physics.dt,
                num_steps=training.eval_steps,
            )

            frozen_entry = {
                "epoch": epoch,
                "k": k,
                **frozen_metrics,
            }
            frozen_log.append(frozen_entry)

            print(
                f"Epoch {epoch:4d} | "
                f"K={k:2d} | "
                f"Frozen worst gap={frozen_metrics['worst_gap']:.4e} | "
                f"Frozen worst residual={frozen_metrics['worst_residual_norm']:.4e}"
            )

    return {
        "online_training": online_log,
        "frozen_evaluation": frozen_log,
    }



## 8. 绘图与结果保存

每次运行只生成当前所选方案的结果目录。

主要输出：

| 文件 | 内容 |
|---|---|
| `training_states.png` | 初值附近的训练数据分布 |
| `training_and_frozen_metrics.png` | 在线训练 gap 与周期性冻结评估指标 |
| `final_comparison_from_y0.png` | 从原始初值出发，MLP 与 Newton 的最终对比 |
| `optimization_report.json` | 配置、训练日志和最终数值指标 |
| `mlp_optimizer_state_dict.pt` | 训练后的网络参数 |

最终对比统一从：

$$
y^{(0)}=y_0
$$

出发。Newton 法作为解析基准。


In [ ]:

def safe_positive(values: List[float]) -> List[float]:
    return [max(float(value), PLOT_FLOOR) for value in values]


def plot_training_metrics(
    training_logs: Dict[str, List[Dict[str, float]]],
    save_path: Path,
) -> None:
    online = training_logs["online_training"]
    frozen = training_logs["frozen_evaluation"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(
        [item["epoch"] for item in online],
        safe_positive(
            [item["worst_trajectory_final_gap"] for item in online]
        ),
        label="Worst online trajectory-final gap",
    )
    axes[0].plot(
        [item["epoch"] for item in online],
        safe_positive(
            [item["mean_trajectory_final_gap"] for item in online]
        ),
        label="Mean online trajectory-final gap",
    )
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Energy gap")
    axes[0].set_title("Online training metrics")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(
        [item["epoch"] for item in frozen],
        safe_positive([item["worst_gap"] for item in frozen]),
        marker="o",
        label="Frozen worst gap",
    )
    axes[1].plot(
        [item["epoch"] for item in frozen],
        safe_positive(
            [item["worst_residual_norm"] for item in frozen]
        ),
        marker="s",
        label="Frozen worst residual norm",
    )
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Metric value")
    axes[1].set_title("Periodic frozen evaluation")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_final_comparison(
    mlp_rollout: List[Dict[str, float]],
    newton_rollout: List[Dict[str, float]],
    save_path: Path,
) -> None:
    steps = [item["step"] for item in mlp_rollout]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(
        steps,
        safe_positive([item["gap"] for item in mlp_rollout]),
        marker="o",
        label="MLP optimizer",
    )
    axes[0, 0].plot(
        steps,
        safe_positive([item["gap"] for item in newton_rollout]),
        marker="s",
        linestyle="--",
        label="Newton method",
    )
    axes[0, 0].set_yscale("log")
    axes[0, 0].set_title("Energy gap from original initial point")
    axes[0, 0].set_xlabel("Iteration")
    axes[0, 0].set_ylabel("E(y) - E(y*)")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(
        steps,
        safe_positive(
            [item["residual_norm"] for item in mlp_rollout]
        ),
        marker="o",
        label="MLP optimizer",
    )
    axes[0, 1].plot(
        steps,
        safe_positive(
            [item["residual_norm"] for item in newton_rollout]
        ),
        marker="s",
        linestyle="--",
        label="Newton method",
    )
    axes[0, 1].set_yscale("log")
    axes[0, 1].set_title("Stationarity residual from original initial point")
    axes[0, 1].set_xlabel("Iteration")
    axes[0, 1].set_ylabel("||grad E(y)||_2")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(
        steps,
        safe_positive(
            [item["position_error"] for item in mlp_rollout]
        ),
        marker="o",
        label="MLP optimizer",
    )
    axes[1, 0].plot(
        steps,
        safe_positive(
            [item["position_error"] for item in newton_rollout]
        ),
        marker="s",
        linestyle="--",
        label="Newton method",
    )
    axes[1, 0].set_yscale("log")
    axes[1, 0].set_title("Position error from original initial point")
    axes[1, 0].set_xlabel("Iteration")
    axes[1, 0].set_ylabel("||y - y*||_2")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(
        steps,
        safe_positive([item["update_norm"] for item in mlp_rollout]),
        marker="o",
        label="MLP optimizer",
    )
    axes[1, 1].plot(
        steps,
        safe_positive(
            [item["update_norm"] for item in newton_rollout]
        ),
        marker="s",
        linestyle="--",
        label="Newton method",
    )
    axes[1, 1].set_yscale("log")
    axes[1, 1].set_title("Update magnitude from original initial point")
    axes[1, 1].set_xlabel("Iteration")
    axes[1, 1].set_ylabel("||delta y||_2")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)



## 9. `main()`：选择并运行一个方案

`main()` 不做批量循环。

运行流程：

1. 根据字符串名称读取一个候选配置；
2. 构造固定物理问题；
3. 生成固定局部扰动训练集；
4. 计算输入归一化统计量；
5. 创建 MLP 和所选 PyTorch 优化器；
6. 训练；
7. 从原始初值 $y_0$ 出发，对比 MLP 与 Newton；
8. 保存图片、JSON 报告和模型参数。

要切换优化器，只需修改最后一个单元格中的：

```python
SELECTED_EXPERIMENT = "sgd_lr_1e-2"
```

或者：

```python
SELECTED_EXPERIMENT = "adam_lr_1e-4"
```


In [ ]:

def main(
    selected_experiment: str,
    *,
    physics: PhysicsConfig = PHYSICS_CONFIG,
    training: TrainingConfig = TRAINING_CONFIG,
    output_root: Path = OUTPUT_ROOT,
) -> Dict[str, object]:
    """选择一个候选方案并执行训练、评估和结果保存。"""
    if selected_experiment not in EXPERIMENTS:
        available = ", ".join(EXPERIMENTS)
        raise ValueError(
            f"Unknown experiment {selected_experiment!r}. "
            f"Available choices: {available}"
        )

    experiment = EXPERIMENTS[selected_experiment]

    # 固定随机种子，使不同候选方案具有相同网络初始化。
    torch.manual_seed(training.model_seed)

    output_dir = output_root / selected_experiment
    output_dir.mkdir(parents=True, exist_ok=True)

    p_n = torch.tensor(physics.p_n, dtype=TORCH_DTYPE)
    v_n = torch.tensor(physics.v_n, dtype=TORCH_DTYPE)

    y0 = p_n.clone()
    y_star = exact_solution(p_n, v_n, physics.g, physics.dt)

    history = torch.cat([p_n, v_n])
    params = torch.tensor(
        [physics.m, physics.g, physics.dt],
        dtype=TORCH_DTYPE,
    )

    training_states = make_training_states_near_initial(
        y0=y0,
        perturbation_std=training.perturbation_std,
        num_perturbation_points=training.num_perturbation_points,
        seed=training.perturbation_seed,
    )

    input_mean, input_std = compute_input_normalizer(
        training_states=training_states,
        history=history,
        params=params,
    )

    model = MLPOptimizer(
        input_mean=input_mean,
        input_std=input_std,
        use_input_normalization=training.use_input_normalization,
        use_dt_scaling=training.use_dt_scaling,
    )

    optimizer = create_optimizer(model, experiment)

    E_star = float(
        variational_energy(
            y_star,
            p_n,
            v_n,
            physics.m,
            physics.g,
            physics.dt,
        ).item()
    )

    newton_solution = y0 + newton_direction(
        y0,
        p_n,
        v_n,
        physics.m,
        physics.g,
        physics.dt,
    )

    print("=" * 72)
    print("Selected experiment:", experiment.name)
    print("Description:", experiment.description)
    print("Output directory:", output_dir)
    print("dtype:", torch.get_default_dtype())
    print("y0:", y0.tolist())
    print("y*:", y_star.tolist())
    print("Newton one-step solution:", newton_solution.tolist())
    print("E*:", E_star)
    print("Training states:", len(training_states))
    print("=" * 72)

    plot_training_states(
        training_states=training_states,
        y0=y0,
        y_star=y_star,
        save_path=output_dir / "training_states.png",
    )

    training_logs = train_selected_model(
        model=model,
        optimizer=optimizer,
        training_states=training_states,
        history=history,
        params=params,
        y_star=y_star,
        E_star=E_star,
        p_n=p_n,
        v_n=v_n,
        physics=physics,
        training=training,
    )

    mlp_rollout = evaluate_rollout(
        model=model,
        initial_y=y0,
        history=history,
        params=params,
        y_star=y_star,
        E_star=E_star,
        p_n=p_n,
        v_n=v_n,
        m=physics.m,
        g=physics.g,
        dt=physics.dt,
        num_steps=training.final_test_steps,
    )

    newton_rollout = evaluate_newton_rollout(
        initial_y=y0,
        y_star=y_star,
        E_star=E_star,
        p_n=p_n,
        v_n=v_n,
        m=physics.m,
        g=physics.g,
        dt=physics.dt,
        num_steps=training.final_test_steps,
    )

    plot_training_metrics(
        training_logs=training_logs,
        save_path=output_dir / "training_and_frozen_metrics.png",
    )

    plot_final_comparison(
        mlp_rollout=mlp_rollout,
        newton_rollout=newton_rollout,
        save_path=output_dir / "final_comparison_from_y0.png",
    )

    final_mlp = mlp_rollout[-1]
    final_newton = newton_rollout[-1]

    report = {
        "selected_experiment": asdict(experiment),
        "physics_config": asdict(physics),
        "training_config": asdict(training),
        "dtype": str(torch.get_default_dtype()),
        "output_directory": str(output_dir),
        "derived_quantities": {
            "y0": y0.tolist(),
            "y_star": y_star.tolist(),
            "newton_one_step_solution": newton_solution.tolist(),
            "E_star": E_star,
            "num_training_states": len(training_states),
            "training_states": [state.tolist() for state in training_states],
            "input_mean": input_mean.tolist(),
            "input_std": input_std.tolist(),
        },
        "training_logs": training_logs,
        "final_rollout_from_y0": {
            "mlp": mlp_rollout,
            "newton": newton_rollout,
        },
        "summary": {
            "mlp_final_gap": final_mlp["gap"],
            "mlp_final_residual_norm": final_mlp["residual_norm"],
            "mlp_final_position_error": final_mlp["position_error"],
            "newton_final_gap": final_newton["gap"],
            "newton_final_residual_norm": final_newton["residual_norm"],
            "newton_final_position_error": final_newton["position_error"],
        },
    }

    report_path = output_dir / "optimization_report.json"
    with report_path.open("w", encoding="utf-8") as file:
        json.dump(report, file, indent=2, ensure_ascii=False)

    model_path = output_dir / "mlp_optimizer_state_dict.pt"
    torch.save(model.state_dict(), model_path)

    print()
    print("Final MLP metrics:")
    print(json.dumps(report["summary"], indent=2, ensure_ascii=False))
    print()
    print("Saved report:", report_path)
    print("Saved model:", model_path)

    return report



## 10. 选择一个方案并运行

默认选择 SGD。要测试 Adam，只需修改字符串。

完整配置会运行 1000 个 epoch。为了快速检查代码结构，可以临时使用：

```python
debug_training = replace(
    TRAINING_CONFIG,
    epochs=10,
    eval_interval=2,
    final_test_steps=5,
)
result = main(
    SELECTED_EXPERIMENT,
    training=debug_training,
)
```

正式实验时使用下面代码中的默认配置即可。


In [ ]:

# 可选值：
#     "sgd_lr_1e-2"
#     "adam_lr_1e-4"
SELECTED_EXPERIMENT = "sgd_lr_1e-2"

result = main(SELECTED_EXPERIMENT)



## 11. 如何对比两个候选方案

Notebook 每次只运行一个候选方案。依次运行两次后，可以比较：

```text
learned_optimizer_float64_teaching_outputs/
├── sgd_lr_1e-2/
│   ├── optimization_report.json
│   ├── mlp_optimizer_state_dict.pt
│   ├── training_states.png
│   ├── training_and_frozen_metrics.png
│   └── final_comparison_from_y0.png
└── adam_lr_1e-4/
    ├── optimization_report.json
    ├── mlp_optimizer_state_dict.pt
    ├── training_states.png
    ├── training_and_frozen_metrics.png
    └── final_comparison_from_y0.png
```

重点观察：

1. `training_and_frozen_metrics.png` 中的周期性冻结评估是否稳定；
2. `final_comparison_from_y0.png` 中的 residual 是否持续下降；
3. MLP 最终 residual 与 Newton residual 的差距；
4. MLP 最终位置误差；
5. 在接近最优解后，更新步长是否继续减小。
